# 02 - Silver Staging: Tratamento de qualidade dos dados Bronze

## Objetivo

Criar uma camada intermediária entre a Bronze e a Silver oficial.

A Silver Staging tem como único objetivo tratar problemas de qualidade de dado sem aplicar regras de negócio.

As operações realizadas nesta etapa são:

- remoção de espaços no início e no final de todas as colunas string;
- substituição de valores textuais sem significado (`Zerados`, `Em Branco`) por `null`;
- substituição de strings vazias por `null`.

Nenhuma transformação de negócio é aplicada aqui.

A tabela gerada é:

`afastamento_inss.silver.stg_beneficios_concedidos`

## 1. Importações

- `col` — referência a colunas do DataFrame;
- `when` — expressão condicional para substituição de valores;
- `trim` — remoção de espaços no início e no final de strings;
- `lit` — valor constante para metadados;
- `current_timestamp` — timestamp da execução.

In [0]:
# ============================================================
# IMPORTACOES
# ============================================================

from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    lit,
    current_timestamp,
    desc
)


## 2. Parâmetros

- `TABELA_ORIGEM` — tabela Bronze gerada no notebook anterior;
- `TABELA_DESTINO` — tabela Silver Staging que será criada.

In [0]:
TABELA_ORIGEM  = "afastamento_inss.bronze.beneficios_concedidos"
TABELA_DESTINO = "afastamento_inss.silver.stg_beneficios_concedidos"

print(f"Origem : {TABELA_ORIGEM}")
print(f"Destino: {TABELA_DESTINO}")


## 3. Leitura da camada Bronze

A Silver Staging sempre lê da Bronze e nunca do arquivo CSV diretamente.

Isso garante que a cadeia de transformações seja rastreável e que qualquer reprocessamento parta do mesmo ponto de entrada.

In [0]:
df_bronze = spark.table(TABELA_ORIGEM)

print(f"Linhas : {df_bronze.count()}")
print(f"Colunas: {len(df_bronze.columns)}")


## 4. Identificação das colunas string

Apenas colunas do tipo string receberão o tratamento de trim e substituição de valores vazios.

As colunas de metadados técnicos com prefixo `_` são excluídas do tratamento porque representam informações de controle geradas pelo pipeline.

In [0]:
colunas_string = [
    campo.name
    for campo in df_bronze.schema.fields
    if campo.dataType.simpleString() == "string"
    and not campo.name.startswith("_")
]

print(f"Colunas string identificadas: {len(colunas_string)}")
for c in colunas_string:
    print(f"  -> {c}")


## 5. Tratamento de qualidade

Para cada coluna string são aplicadas três transformações em sequência:

1. `trim` — remove espaços no início e no final;
2. string vazia após trim → `null`;
3. valores textuais sem significado → `null`:
   - `Zerados` — valor padrão do INSS para campos não preenchidos;
   - `Em Branco` — valor padrão do INSS para campos sem informação.

A substituição por `null` é intencional porque:

- permite uso de funções nativas do Spark como `isNull()` e `dropna()`;
- facilita contagem de ausência de informação na Silver Oficial;
- evita que análises de frequência incluam esses valores como categor

In [0]:
SENTINELAS_CONHECIDOS = [
    "0", "", "{ñ class}", "{Ñ class}",
    "Zerados", "Em Branco",
    "zerados", "em branco",
    "Nao Informado", "nao informado",
    "N/A", "NA", "n/a", "-", "."
]


print("=" * 60)
print("DIAGNOSTICO COMPLETO - TOP 5 VALORES POR COLUNA")
print("=" * 60)

for c in colunas_string:
    print(f"\n[COLUNA] {c}")
    print("-" * 40)

    top5 = (
        df_bronze
        .groupBy(c)
        .count()
        .orderBy(desc("count"))
        .limit(5)
    )

    for row in top5.collect():
        valor = row[c]
        qtd   = row["count"]

        if valor is None:
            status = "[NULL]"
        elif str(valor).strip() in SENTINELAS_CONHECIDOS:
            status = "[SENTINEL]"
        else:
            status = ""

        print(f"  {str(valor):45} {qtd:>8,}  {status}")

print("\n" + "=" * 60)
print("FIM DO DIAGNOSTICO")
print("=" * 60)

In [0]:
# Registros onde cid_cod e null mas cid_desc tem valor

caso_a = df_bronze.filter(
    col("cid_cod").isNull() & col("cid_desc").isNotNull()
).count()

# Registros onde cid_desc e null mas cid_cod tem valor
caso_b = df_bronze.filter(
    col("cid_desc").isNull() & col("cid_cod").isNotNull()
).count()

# Registros onde ambos sao null
ambos_null = df_bronze.filter(
    col("cid_cod").isNull() & col("cid_desc").isNull()
).count()

print("=" * 50)
print("INVESTIGACAO DIVERGENCIA CID")
print("=" * 50)
print(f"cid_cod null, cid_desc preenchido : {caso_a:>8,}")
print(f"cid_desc null, cid_cod preenchido : {caso_b:>8,}")
print(f"Ambos null                        : {ambos_null:>8,}")
print("=" * 50)

# Amostra do caso B para entender o padrao
if caso_b > 0:
    print("\nAmostra - cid_cod preenchido mas cid_desc null:")
    display(
        df_bronze.filter(
            col("cid_desc").isNull() & col("cid_cod").isNotNull()
        )
        .select("especie_desc", "cid_cod", "cid_desc")
        .limit(10)
    )

In [0]:
# ============================================================
# REGRAS DE TRATAMENTO DE QUALIDADE - SILVER STAGING
# ============================================================

# Sentinelas gerais aplicados em todas as colunas string
SENTINELAS_GERAIS = [
    "Zerados",
    "Em Branco",
    "zerados",
    "em branco",
    "{ñ class}",
    "{Ñ class}"
]

# Colunas onde "0" representa ausencia de informacao
# despacho_cod   EXCLUIDO - "0" = Concessao Normal (valido)
# qt_anos_contribuicao EXCLUIDO - "0" = zero anos (valido)
COLUNAS_COM_ZERO_COMO_SENTINEL = [
    "cid_cod",
    "cid_desc",
    "cnae_2023"
]

# Colunas com sentinel "Nao Informado"
COLUNAS_NAO_INFORMADO = [
    "vinculo_dependentes",
    "grau_instrucao"
]

# Colunas com sentinel especifico de municipio zerado
COLUNAS_MUNICIPIO = [
    "mun_resid"
]
SENTINEL_MUNICIPIO = "00000-Zerada"

# ============================================================
# APLICACAO DO TRATAMENTO
# ============================================================

df_staging = df_bronze

for c in colunas_string:
    sentinelas_coluna = SENTINELAS_GERAIS.copy()

    if c in COLUNAS_COM_ZERO_COMO_SENTINEL:
        sentinelas_coluna.append("0")

    if c in COLUNAS_NAO_INFORMADO:
        sentinelas_coluna.append("Não Informado")
        sentinelas_coluna.append("Nao Informado")

    if c in COLUNAS_MUNICIPIO:
        sentinelas_coluna.append(SENTINEL_MUNICIPIO)

    df_staging = df_staging.withColumn(
        c,
        when(
            trim(col(c)).isin(sentinelas_coluna) | (trim(col(c)) == ""),
            None
        ).otherwise(trim(col(c)))
    )

print("Tratamento aplicado.")
print(f"Total de colunas tratadas              : {len(colunas_string)}")
print(f"Colunas com sentinel zero              : {len(COLUNAS_COM_ZERO_COMO_SENTINEL)}")
print(f"Colunas com sentinel Nao Informado     : {len(COLUNAS_NAO_INFORMADO)}")
print(f"Colunas com sentinel municipio zerado  : {len(COLUNAS_MUNICIPIO)}")


In [0]:
# ============================================================
# VERIFICACAO POS-TRATAMENTO - SILVER STAGING
# ============================================================
# Mapa de verificacao: coluna -> sentinel que deve ter sumido
VERIFICACOES = {
    "cid_cod"                  : ["0", "Zerados", "Em Branco"],
    "cid_desc"                 : ["0", "Zerados", "Em Branco"],
    "cnae_2023"                : ["0"],
    "cnae_2024"                : ["{ñ class}", "{Ñ class}"],
    "pais_acordo_internacional": ["{ñ class}", "{Ñ class}"],
    "vinculo_dependentes"      : ["Não Informado", "Nao Informado"],
    "grau_instrucao"           : ["Não Informado", "Nao Informado"],
    "mun_resid"                : ["00000-Zerada"]
}

print("=" * 65)

print("VERIFICACAO POS-TRATAMENTO")
print("=" * 65)

print(f"  {'COLUNA':<35} {'SENTINEL':<20} {'RESULT'}")
print("-" * 65)

erros = 0

for coluna, sentinelas in VERIFICACOES.items():
    for sentinel in sentinelas:
        restantes = df_staging.filter(
            col(coluna) == sentinel
        ).count()

        status = "OK" if restantes == 0 else "FALHA"
        if restantes > 0:
            erros += 1

        print(f"  {coluna:<35} {sentinel:<20} {status} ({restantes:,})")

print("-" * 65)

# Verificacao de valores nulos resultantes
print("\n")
print("=" * 65)

print("NULOS RESULTANTES POR COLUNA")
print("=" * 65)

print(f"  {'COLUNA':<35} {'NULOS':>10} {'PERCENTUAL':>12}")
print("-" * 65)

total = df_staging.count()

for coluna in VERIFICACOES.keys():
    nulos = df_staging.filter(col(coluna).isNull()).count()
    pct   = (nulos / total) * 100
    print(f"  {coluna:<35} {nulos:>10,} {pct:>11.2f}%")

print("-" * 65)
print(f"\nTotal de registros : {total:,}")
print(f"Verificacoes com falha : {erros}")

if erros == 0:
    print("RESULTADO: Todos os sentinelas foram convertidos corretamente.")
else:
    print("RESULTADO: ATENCAO - Existem sentinelas nao tratados.")

print("=" * 65)


## 6. Atualização dos metadados técnicos

A coluna `_data_ingestao` é atualizada para registrar o momento de processamento da Silver Staging, diferenciando-o do momento de ingestão da Bronze.

In [0]:
df_staging = df_staging.withColumn(
    "_data_ingestao",
    current_timestamp()
)


## 7. Validação pré-gravação

Antes de gravar a tabela, é verificado o impacto do tratamento aplicado.

A comparação mostra quantos valores foram convertidos para `null` em cada coluna, confirmando que o tratamento foi aplicado corretamente.

In [0]:
contagem_nulos_antes = df_bronze.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in colunas_string
])

contagem_nulos_depois = df_staging.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in colunas_string
])

print("Nulos ANTES do tratamento:")
display(contagem_nulos_antes)

print("Nulos DEPOIS do tratamento:")
display(contagem_nulos_depois)


## 8. Gravação da Silver Staging

A tabela é gravada no schema `silver` do catálogo `afastamento_inss`.

O modo `overwrite` permite reprocessamento integral sempre que necessário.

In [0]:
(
    df_staging
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DESTINO)
)

print(f"Tabela gravada: {TABELA_DESTINO}")


## 9. Validação da carga

Comparação entre Bronze e Silver Staging para garantir que nenhum registro foi perdido.

In [0]:
linhas_bronze  = df_bronze.count()
linhas_staging = spark.table(TABELA_DESTINO).count()

print("=" * 45)
print("VALIDAÇÃO SILVER STAGING")
print("=" * 45)
print(f"{'':25} {'BRONZE':>8} {'STAGING':>8}")
print("-" * 45)
print(f"{'Linhas':25} {linhas_bronze:>8,} {linhas_staging:>8,}")
print("-" * 45)

if linhas_bronze == linhas_staging:
    print("Linhas: OK — nenhum registro perdido")
else:
    diff = linhas_bronze - linhas_staging
    print(f"Linhas: DIVERGÊNCIA de {diff:,} registros")

print("=" * 45)

display(spark.table(TABELA_DESTINO).limit(5))


## Conclusão

A tabela `afastamento_inss.silver.stg_beneficios_concedidos` foi criada com sucesso.

| Item | Valor |
|---|---|
| Origem | afastamento_inss.bronze.beneficios_concedidos |
| Linhas processadas | 463.568 |
| Colunas string tratadas | 27 |
| Valores substituídos por null | Zerados, Em Branco, string vazia |

### Decisões desta etapa

| Decisão | Justificativa |
|---|---|
| Trim em todas as strings | Eliminar espaços invisíveis herdados do CSV |
| Zerados → null | Representação técnica de ausência de informação |
| Em Branco → null | Representação técnica de ausência de informação |
| String vazia → null | Uniformizar ausência de dado |
| Metadados Bronze preservados | Rastreabilidade até o arquivo de origem |

### Próxima etapa

`03_silver_oficial`

Aplicar tipagem, separação de código e descrição, classificação de CID e espécie de benefício a partir da Silver Staging.